# Phase 2 — MOT-oriented training ideas (standard transformer head)

This notebook keeps the **original 5-D output** (`MotionTransformer` + `LossFunction`) and layers practical changes that often help trackers:

1. **Temporal discontinuity sampling** — `random_jump=True` samples two disjoint segments from a longer window so the model sees harder context switches (crowd / camera motion).
2. **Synthetic occlusion / misses** — `random_drop_prob` in `GTSequenceDataset` zeros bbox + motion channels and confidence on random past frames (now active in `dataset.py`).
3. **Focal-style CIoU** — up-weight examples with low CIoU so hard trajectories contribute more.
4. **Velocity consistency** — penalize mismatch between predicted first differences and GT velocities (smoother, more physically plausible tracks).
5. **Multi-dataset** — uncomment paths to mix MOT17 / MOT20 / DanceTrack / SportsMOT for domain diversity.

These are complementary to learned Q/R notebooks: you can train a strong point predictor here, then distill uncertainties or use its features inside a graph tracker.

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader

from dataset import GTSequenceDataset
from transformer_encoder import MotionTransformer
from loss import LossFunction

SEQ_IN_LEN = 30
SEQ_OUT_LEN = 20
SEQ_TOTAL_LEN = 80
BATCH_SIZE = 512
STEPS = 4
NOISE_COEFFICIENT = 0.15
NOISE_PROB = 0.2
RANDOM_DROP_PROB = 0.08
BASE_DIR = os.environ.get("MOT_DATASET_ROOT", "../../Datasets/")

train_roots = [
    f"{BASE_DIR}MOT17/train",
    # f"{BASE_DIR}MOT20/train",
    # f"{BASE_DIR}DanceTrack/train",
    # f"{BASE_DIR}SportsMOT/train",
]
val_roots = [
    f"{BASE_DIR}MOT17/val",
]

train_dataset = GTSequenceDataset.from_roots(
    train_roots,
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    random_jump=True,
    noise_coeff=NOISE_COEFFICIENT,
    noise_prob=NOISE_PROB,
    random_drop_prob=RANDOM_DROP_PROB,
)
val_dataset = GTSequenceDataset.from_roots(
    val_roots,
    seq_in_len=SEQ_IN_LEN,
    seq_out_len=SEQ_OUT_LEN,
    seq_total_len=SEQ_TOTAL_LEN,
    steps=STEPS,
    random_jump=False,
    noise_coeff=NOISE_COEFFICIENT,
    noise_prob=NOISE_PROB,
    random_drop_prob=0.0,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train {len(train_dataset)} | Val {len(val_dataset)}")

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LR = 4e-4
NUM_EPOCHS = 25
FOCAL_GAMMA = 1.0
VEL_COEFF = 0.15
SMOOTH_COEFF = 0.35

model = MotionTransformer(
    input_dim=13,
    output_dim=5,
    d_model=256,
    nhead=8,
    num_layers=4,
    dim_ff=1024,
    dropout=0.1,
).to(DEVICE)

ciou_helper = LossFunction()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(NUM_EPOCHS, 1))


def advanced_loss(pred, gt_future):
    """pred, gt_future: (B, T-1, 5+) aligned like train_one_epoch."""
    pb, gb = pred[:, :, :4], gt_future[:, :, :4]
    c = ciou_helper.ciou(gb, pb)
    one_m = (1.0 - c).clamp(min=0.0)
    focal_ciou = (one_m.pow(FOCAL_GAMMA) * one_m).mean()
    smooth = F.smooth_l1_loss(pb, gb)
    conf = F.smooth_l1_loss(pred[:, :, 4:5], c.detach().unsqueeze(-1))
    if pred.size(1) > 1:
        pv = pred[:, 1:, :4] - pred[:, :-1, :4]
        gv = gt_future[:, 1:, :4] - gt_future[:, :-1, :4]
        vel = F.smooth_l1_loss(pv, gv)
    else:
        vel = pred.new_tensor(0.0)
    return focal_ciou + SMOOTH_COEFF * smooth + conf + VEL_COEFF * vel


def train_one_epoch():
    model.train()
    tot = 0.0
    for src, trg, _, gt_trg in train_loader:
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        gt_trg = gt_trg.to(DEVICE)
        optimizer.zero_grad()
        out = model(src, trg[:, :-1])
        loss = advanced_loss(out, gt_trg[:, 1:])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot += loss.item()
    return tot / max(len(train_loader), 1)


@torch.no_grad()
def evaluate():
    model.eval()
    tot = 0.0
    for src, trg, _, gt_trg in val_loader:
        src = src.to(DEVICE)
        trg = trg.to(DEVICE)
        gt_trg = gt_trg.to(DEVICE)
        out = model(src, trg[:, :-1])
        tot += advanced_loss(out, gt_trg[:, 1:]).item()
    return tot / max(len(val_loader), 1)


print(f"model params {sum(p.numel() for p in model.parameters()):,} | device={DEVICE}")

In [ ]:
best = float("inf")
os.makedirs("pretrained", exist_ok=True)
ckpt = "pretrained/transformer_mot_advanced.pth"

for ep in range(1, NUM_EPOCHS + 1):
    tr = train_one_epoch()
    va = evaluate()
    scheduler.step()
    if va < best:
        best = va
        model.save_weight(ckpt)
    print(f"epoch {ep:03d}  train {tr:.5f}  val {va:.5f}")

print("best:", best, "->", ckpt)